# Fafnir Playground

In [ ]:
from jormungandr.fafnir import Fafnir
from jormungandr.backbone import Backbone

In [ ]:
backbone = Backbone()
fafnir = Fafnir(backbone=backbone, num_queries=100, encoder_type="mamba")
detr   = Fafnir(backbone=backbone, num_queries=100, encoder_type="detr")

## 

In [ ]:
import requests
from PIL import Image
from io import BytesIO
import torch
from torchvision import transforms

# Load image from URL
url ="http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(url)
img = Image.open(BytesIO(response.content)).convert("RGB")

# Convert to tensor



transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

tensor = transform(img)
print(tensor.shape)  # (C, H, W)


In [ ]:
print("Running forward pass through DETR...")
detr_class_labels, detr_bbox = detr.forward(tensor.unsqueeze(0))  # Add batch dimension
print("DETR Class Labels Shape:", detr_class_labels.shape)
print("DETR BBox Shape:", detr_bbox.shape)

print("Running forward pass through Fafnir...")
fafnir_class_labels, fafnir_bbox = fafnir.forward(tensor.unsqueeze(0))  # Add batch dimension
print("Fafnir Class Labels Shape:", fafnir_class_labels.shape)
print("Fafnir BBox Shape:", fafnir_bbox.shape)

In [ ]:
detr_class_labels[0][0]  # Show first 5 class label predictions from DETR

In [ ]:
fafnir_class_labels[0][0]  # Show first 5 class label predictions from Fafnir

In [ ]:
# Display results
from transformers import DetrImageProcessor
from dataclasses import dataclass

@dataclass
class DetrOutput:
    logits: torch.Tensor
    pred_boxes: torch.Tensor

processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
results = processor.post_process_object_detection(
    outputs=DetrOutput(logits=detr_class_labels, pred_boxes=detr_bbox),
    target_sizes=[(img.height, img.width)], threshold=0.3
)

In [ ]:
results